# LMP spread - how much is the zone-wide part?

Share of each node's spread variance explained by the same-hour mean spread of its zone (an oracle, since it uses realized values). `mae_ratio` is the residual MAE over the spread MAE.

In [ ]:
import numpy as np
import pandas as pd

from lmp_model.dataset import load_cache, load_nodes

nodes = load_nodes()
data = load_cache()

In [ ]:
yy = np.where(data["exists"], data["y"], np.nan)
rows = []
for zone, idx in nodes.reset_index().groupby("zone").indices.items():
    if len(idx) < 3:
        continue
    Y = yy[:, idx]
    resid = Y - np.nanmean(Y, axis=1, keepdims=True)
    ss_tot = np.nansum((Y - np.nanmean(Y, axis=0)) ** 2)
    ss_res = np.nansum((resid - np.nanmean(resid, axis=0)) ** 2)
    rows.append({"zone": zone, "nodes": len(idx), "r2_zone_mean": 1 - ss_res / ss_tot,
                 "mae_ratio": np.nanmean(np.abs(resid)) / np.nanmean(np.abs(Y))})
pd.DataFrame(rows).set_index("zone").sort_values("r2_zone_mean").round(3)